In [2]:
import numpy as np
import pandas as pd

train_df = pd.read_csv("data/IMT2024024_train_var1.csv")
test_df  = pd.read_csv("data/IMT2024024_test_var1.csv")

print(f"Train Dataset Shape: {train_df.shape}")
print(f"Test Dataset Shape:  {test_df.shape}")
print("\nFirst 5 rows of Training Data:")
print(train_df.head())

Train Dataset Shape: (1000, 7)
Test Dataset Shape:  (1000, 6)

First 5 rows of Training Data:
         x1        x2        x3        x4        x5        x6         y
0 -1.000000 -0.514132  0.103341 -0.144700  0.575898  0.785434  0.976873
1 -0.304260 -1.000000 -0.779001 -0.148084 -1.000000 -1.000000  1.621223
2  0.079625 -1.000000 -0.694604  0.068184  0.653448  0.841697 -0.081755
3  1.000000  1.000000  0.268996  0.277195  0.186076  0.714304 -0.456919
4 -0.510508 -0.982768 -0.566677  0.461428  0.689030  0.429986 -0.567492


In [3]:
feature_names = [col for col in train_df.columns if col != 'y']

X_train_raw = train_df[feature_names].to_numpy()
y_train = train_df['y'].to_numpy().reshape(-1, 1)
X_test_raw  = test_df[feature_names].to_numpy()

print(f"X_train shape: {X_train_raw.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_test shape:  {X_test_raw.shape}")

X_train shape: (1000, 6)
y_train shape: (1000, 1)
X_test shape:  (1000, 6)


In [4]:
def fit_ridge(X_design, y, lam=1e-3):                   
    """
    Solves: w = (X^T @ X + lam * N * I)^(-1) @ X^T @ y
    """
    N, D = X_design.shape
    I = np.eye(D)
    I[0, 0] = 0.0  
    
    # np.linalg.solve solves A w = b directly without computing an explicit inverse
    w = np.linalg.solve(X_design.T @ X_design + lam * N * I, X_design.T @ y)
    return w

def predict(X_design, w):
    return X_design @ w

def compute_mse(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)

def compute_r2(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    return 1.0 - (ss_res / ss_tot)

In [ ]:
from sklearn.model_selection import KFold
from sklearn.preprocessing import PolynomialFeatures

X = np.asarray(X_train_raw)
y = np.asarray(y_train)

degrees = list(range(1, 11))
# lambdas = [1e-4, 1e-3, 1e-2, 0.1, 0, 1.0, 10.0]
# here I first ran over the below lambda values and got 0.01 as the best lambda with degree around degree 5

# lambdas = [0.003, 0.004, 0.005, 0.006, 0.008]
# then I grdually found it to be around 0.004 

# so I checked again with this...
# lambdas = np.linspace(0.0025, 0.0055, 31)

lambdas = [0.0033 ,0.0034, 0.0035, 0.0036, 0.0037]

kf = KFold(n_splits=5, shuffle=True, random_state=42)

grid_results = []
best_val_mse = float("inf")
best_degree = None
best_lambda = None
best_val_r2 = None

print("Running 5-Fold Cross-Validation Grid Search...")
print(f"Degrees: {degrees}")
print(f"Lambdas: {lambdas}")
print("-" * 75)

for d in degrees:
    metrics_per_lambda = {lam: {'train_mse': [], 'val_mse': [], 'val_r2': []} for lam in lambdas}         

    for train_idx, val_idx in kf.split(X_train_raw):
        
        X_train_fold, X_val_fold = X_train_raw[train_idx], X_train_raw[val_idx]
        y_train_fold, y_val_fold = y_train[train_idx], y_train[val_idx]

        poly = PolynomialFeatures(degree=d, include_bias=True)
        X_train_poly = poly.fit_transform(X_train_fold)
        X_val_poly   = poly.transform(X_val_fold)

        N_fold, D_fold = X_train_poly.shape
        XT_X = X_train_poly.T @ X_train_poly
        XT_y = X_train_poly.T @ y_train_fold


        I = np.eye(D_fold)
        I[0, 0] = 0.0  

        for lam in lambdas:
            # Solve (X^T X + lam * N * I) w = X^T y
            weights = np.linalg.solve(XT_X + lam * N_fold * I, XT_y)

            y_train_pred = predict(X_train_poly, weights)
            y_val_pred   = predict(X_val_poly, weights)

            metrics_per_lambda[lam]['train_mse'].append(compute_mse(y_train_fold, y_train_pred))
            metrics_per_lambda[lam]['val_mse'].append(compute_mse(y_val_fold, y_val_pred))
            metrics_per_lambda[lam]['val_r2'].append(compute_r2(y_val_fold, y_val_pred))

    for lam in lambdas:
        mean_tr_mse  = np.mean(metrics_per_lambda[lam]['train_mse'])
        mean_val_mse = np.mean(metrics_per_lambda[lam]['val_mse'])
        mean_val_r2  = np.mean(metrics_per_lambda[lam]['val_r2'])

        grid_results.append({
            'degree': d,
            'lambda': lam,
            'train_mse': mean_tr_mse,
            'val_mse': mean_val_mse,
            'val_r2': mean_val_r2,
            'total_features': D_fold
        })

        if mean_val_mse < best_val_mse:
            best_val_mse = mean_val_mse
            best_degree  = d
            best_lambda  = lam
            best_val_r2  = mean_val_r2

    # print(f"Degree {d:2d} finished (tested {len(lambdas)} lambdas across 5 folds).")

results_df = pd.DataFrame(grid_results)

print("\nTop 5 Configurations (Ranked by Validation MSE):")
print(results_df.sort_values(by='val_mse').head(5).to_string(index=False))

print("\n" + "=" * 60)
print(f"Optimal Degree:       {best_degree}")
print(f"Optimal Lambda:       {best_lambda}")
print(f"Best Validation MSE:  {best_val_mse:.4e}")
print(f"Corresponding Val R2: {best_val_r2:.4f}")
print("=" * 60)



Running 5-Fold Cross-Validation Grid Search...
Degrees: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
Lambdas: [0.0033, 0.0034, 0.0035, 0.0036, 0.0037]
---------------------------------------------------------------------------

Top 5 Configurations (Ranked by Validation MSE):
 degree  lambda  train_mse  val_mse   val_r2  total_features
      5  0.0036   0.195357 0.505464 0.947408             462
      5  0.0035   0.193975 0.505464 0.947407             462
      5  0.0037   0.196725 0.505510 0.947405             462
      5  0.0034   0.192578 0.505515 0.947401             462
      5  0.0033   0.191165 0.505621 0.947389             462

Optimal Degree:       5
Optimal Lambda:       0.0036
Best Validation MSE:  5.0546e-01
Corresponding Val R2: 0.9474


In [7]:
final_poly = PolynomialFeatures(degree=best_degree, include_bias=True)
X_train_poly = final_poly.fit_transform(X_train_raw)
X_test_poly  = final_poly.transform(X_test_raw)

final_weights = fit_ridge(X_train_poly, y_train, lam=best_lambda)

y_test_pred = predict(X_test_poly, final_weights).ravel()

submission_df = pd.DataFrame({'y': y_test_pred})

csv_filename = "prediction_files/IMT2024024_pred_var1.csv"
submission_df.to_csv(csv_filename, index=False)

print(f"Predictions saved with only column 'y' to:")
print(f" - CSV:   {csv_filename}")
print("\nFirst 5 predicted rows:")
print(submission_df.head())

Predictions saved with only column 'y' to:
 - CSV:   prediction_files/IMT2024024_pred_var1.csv

First 5 predicted rows:
          y
0  2.405280
1  4.275298
2  1.121099
3 -4.040885
4 -1.954892
